## PRACTICA OBLIGATORIA: **No Supervisado: PCA**

* La práctica obligatoria de esta unidad consiste en aplicar PCA a un dataset de imágenes con diferentes objetivos y compromisos. Descarga este notebook en tu ordenador y trabaja en local. Ten en cuenta que tendrás que descar los directorios de imágenes y datos adicionales, si los hubiera.
* Recuerda que debes subirla a tu repositorio personal antes de la sesión en vivo para que puntúe adecuadamente.  
* Recuerda también que no es necesario que esté perfecta, sólo es necesario que se vea el esfuerzo. 
* Esta práctica se resolverá en la sesión en vivo correspondiente y la solución se publicará en el repo del curso. 

### El problema de negocio

El Caesar Palace de las Vegas está planificando la instalación de mil quininetas microcámaras en los accesos a sus instalaciones para las próximas sesiones del "Poker World Championship". Estas microcámaras tienen la peculiaridad de que son capaces de tomar fotos encuadradas de las caras y la desventaja de que no tienen un gran ancho de banda de comunicación. (Las había de más ancho y de mayor precio...). NOTA: El ancho de banda limita el tamaño de las imágenes que pueden enviar las microcámaras).

El objetivo de las microcámaras es el de detectar personas "non-gratas" en tiempo real, pudiendo posprocesar las imágenes para poder detectar si han accedido a las instalaciones personas que estuvieran perseguidas por la ley, en los bancos de datos de los casinos identificadas como "peligrosas" (no se sabe si para el resto de personas o para los beneficios de los casinos) y en las listas de no admisión de jugadores adictos. Por eso no necesitan procesar los datos en tiempo real, pero sí enviarlos a un repositorio central. 

¿Cuál es su problema? O bien comprimen las imágenes y las procesan comprimidas en cada microcámara (pueden comprimir muy rápido pero no tienen cpu para procesarlas sin comprimir) o bien las comprimen y las mandan a un servidor central muy rápido (por eso ti) donde se descomprimirían y se analizarían. Analizar quiere decir en este contexto, pasarles un modelo de clasificación que determine si la persona de la imagen es una de las listas prohibidas (o sea que clasifique la imágen).  

Nos han enviado un dataset y con él debemos estudiar cuál de las dos soluciones es más interesante y dar recomendaciones al respecto. Vamos a ello.

### Ejercicio 0

Importa los paquetes y módulos que necesites a lo largo del notebook.

In [ ]:
# Importaciones
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# sklearn
from sklearn.datasets import fetch_olivetti_faces
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
plt.style.use('ggplot')

### #1 MODELO DE BASE

**Objetivo:** Construir un modelo baseline de clasficación de imágenes que las trate sin comprimir (es decir usando todos sus píxeles).

Para conseguir el objetivo, primero descarga el dataset de las caras de Olivetti que ya has utilizado anteriormente, empleando las funciones de sklearn necesarias. Luego, construye un clasificador con el modelo que consideres más apropiado y todas las features del dataset. Eso sí, recuerda hacer lo siguiente:

1. Construir un data frame con los datos 
2. Hacer un split en train y test con al menos 80 instancias en el test y estratificado según el target. Este split se ha de mantener en el resto de la práctica
3. Hacer un quick miniEDA o justificar el no hacerlo.
4. Medir la recall media ("balanced_accuracy") sobre cross validation con 5 folds y sobre el conjunto de test y guarda ambas para usarlas como baseline en las siguientes partes


In [ ]:
# Cargamos el dataset de Olivetti faces
faces = fetch_olivetti_faces()

X = faces.data    # shape: (400, 4096)  -> 400 imagenes de 64x64 pixeles aplanadas
y = faces.target  # shape: (400,)       -> 40 personas (0-39)

print(f"Shape X: {X.shape}")
print(f"Shape y: {y.shape}")
print(f"Numero de clases: {len(np.unique(y))}")
print(f"Rango de pixeles: [{X.min():.3f}, {X.max():.3f}]")

# DataFrame con los datos
df = pd.DataFrame(X)
df['target'] = y

print(f"\nDataFrame shape: {df.shape}")
df.head(3)

In [ ]:
# Split train/test
# 0.2 * 400 = 80 instancias en test, que es el minimo pedido
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}  --> {X_test.shape[0]} instancias (>= 80 OK)")
print("\nEste split se mantiene igual para toda la practica")

In [ ]:
# Mini EDA
# Justificacion: Las imagenes ya estan normalizadas en [0,1] por sklearn,
# el dataset esta perfectamente balanceado (10 imgs por persona, 8 en train 2 en test)
# y no hay valores nulos, por lo que no hay mucho que transformar a priori.
# Solo veo unas cuantas caras para confirmar que los datos son correctos.

print("Distribucion de clases en train (primeras 10):")
print(pd.Series(y_train).value_counts().sort_index()[:10])
print("... todas las clases tienen 8 muestras en train")

# Visualizamos algunas caras
fig, axes = plt.subplots(3, 8, figsize=(16, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i].reshape(64, 64), cmap='gray')
    ax.set_title(f"P:{y_train[i]}", fontsize=7)
    ax.axis('off')
plt.suptitle("Muestra del dataset (P = persona/clase)", fontsize=11)
plt.tight_layout()
plt.show()

print(f"\nValores nulos: {df.isnull().sum().sum()}")
print(f"Shape de cada imagen: 64x64 = {64*64} features")

In [ ]:
# Modelo baseline con TODAS las features (los 4096 pixeles)
# Uso RandomForest porque es robusto y funciona bien con muchas features
rf_base = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

# CV con 5 folds y balanced_accuracy
print("Calculando CV baseline (puede tardar un momento)...")
cv_scores_base = cross_val_score(
    rf_base, X_train, y_train,
    cv=5,
    scoring='balanced_accuracy'
)

cv_baseline = cv_scores_base.mean()
print(f"\nScores CV: {cv_scores_base.round(4)}")
print(f"CV balanced_accuracy (media 5 folds): {cv_baseline:.4f}")

# Evaluacion en test
rf_base.fit(X_train, y_train)
test_baseline = balanced_accuracy_score(y_test, rf_base.predict(X_test))
print(f"Test balanced_accuracy:               {test_baseline:.4f}")

print(f"\n--- BASELINE GUARDADO ---")
print(f"CV:   {cv_baseline:.4f}")
print(f"Test: {test_baseline:.4f}")

### #2 MODELO PARA LAS MICROCÁMARAS
**Objetivo:** Construir un modelo que pueda funcionar en las microcámaras, es decir que pueda funcionar con datos comprimidos.

Para cumplir con el objetivo se os ocurre emplear la doble propiedad de la PCA, que permite comprimir datos y mantener la capacidad informativa de estos. Sigue los siguientes pasos:
1. Instancia un objeto PCA sobre los datos de Train sin especificar ni componentes ni varianza explicada (o sea sin pasar argumentos).
2. Escoge un rango de valores para el número de PCAs que permitan por lo menos una compresión de la imagen de entre el 0.2% y el 2.5% (prueba al menos 5 valores). NOTA: La compresión es la reducción total, es decir una reducción del 1% quiere decir que el dataset se reduce a un 1% de su tamaño original)
3. Para el rango anterior entrena un modelo de clasificación y apunta su scoring en una validación cruzada de 5 folds y métrica el recall medio y su scoring contra test.
4. Muestra en un dataframe el valor de numero de componentes principales empleado, el scoring en CV, el scoring contra test, el % de compresión, la diferencia con el scoring de CV del modelo base, la diferencia con el scoring en test.
5. Escoge el número de componentes que permitirían tener la mayor compresión con una pérdida inferior a 3 puntos porcentuales tanto en CV como en test. Si no hay escoge el que tenga una pérdida inferior a 5 puntos porcentuales. 

In [ ]:
# Hay que escalar antes de aplicar PCA (aunque los pixeles ya esten en [0,1])
# Si no se escala la PCA puede estar dominada por las variables con mayor varianza
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)  # ojo, solo transform en test!

# PCA sin especificar numero de componentes
pca_full = PCA()
pca_full.fit(X_train_scaled)

# Transformamos ambos conjuntos
X_train_pca = pca_full.transform(X_train_scaled)
X_test_pca  = pca_full.transform(X_test_scaled)

print(f"X_train_pca shape: {X_train_pca.shape}")
print(f"X_test_pca shape:  {X_test_pca.shape}")
print(f"Total componentes: {pca_full.n_components_}")

In [ ]:
# Analizamos la varianza explicada acumulada para orientarnos
var_acum = pca_full.explained_variance_ratio_.cumsum()

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(range(1, 201), var_acum[:200], 'steelblue')
ax.axhline(0.95, color='red',    linestyle='--', label='95% varianza')
ax.axhline(0.99, color='orange', linestyle='--', label='99% varianza')
ax.set_xlabel('Numero de componentes')
ax.set_ylabel('Varianza explicada acumulada')
ax.set_title('Varianza explicada acumulada - Olivetti Faces')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

for thr in [0.90, 0.95, 0.99]:
    n = np.argmax(var_acum >= thr) + 1
    print(f"Componentes para {thr*100:.0f}% varianza: {n}")

In [ ]:
# Calculamos el rango de componentes para compresion entre 0.2% y 2.5%
# La compresion aqui significa el % que representa el nuevo dataset respecto al original
n_features = X_train.shape[1]  # 4096 features

print(f"Features originales: {n_features}")
print(f"0.2% de {n_features} = {n_features * 0.002:.1f} -> ~8 componentes")
print(f"2.5% de {n_features} = {n_features * 0.025:.1f} -> ~102 componentes")

# Elijo 5 valores distribuidos en ese rango
n_components_p2 = [8, 25, 50, 75, 100]
print(f"\nValores a probar: {n_components_p2}")
print(f"% del original:   {[round(n/n_features*100, 2) for n in n_components_p2]}")

In [ ]:
# Loop: para cada n_components entrenamos y evaluamos
resultados_p2 = []

for n in n_components_p2:
    # Cogemos las primeras n componentes del PCA completo
    X_tr_n = X_train_pca[:, :n]
    X_te_n = X_test_pca[:, :n]

    rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

    # CV 5 folds
    cv_score = np.mean(cross_val_score(rf, X_tr_n, y_train, cv=5, scoring='balanced_accuracy'))

    # Test
    rf.fit(X_tr_n, y_train)
    test_score = balanced_accuracy_score(y_test, rf.predict(X_te_n))

    compresion = (n / n_features) * 100

    resultados_p2.append({
        'n_componentes': n,
        'cv_bal_acc':    round(cv_score, 4),
        'test_bal_acc':  round(test_score, 4),
        'compresion_%':  round(compresion, 3),
        'diff_cv':       round(cv_score - cv_baseline, 4),
        'diff_test':     round(test_score - test_baseline, 4)
    })
    print(f"  n={n:3d} | CV={cv_score:.4f} | Test={test_score:.4f} | Compr={compresion:.2f}%")

In [ ]:
# Resultados en DataFrame
df_p2 = pd.DataFrame(resultados_p2)
print("Tabla de resultados - Parte 2:")
display(df_p2)

# Escogemos el de mayor compresion con perdida < 3pp en CV y test
# (mayor compresion = menor % de tamanho respecto al original = menor n_componentes)
criterio_3 = df_p2[(df_p2['diff_cv'] >= -0.03) & (df_p2['diff_test'] >= -0.03)]

if len(criterio_3) > 0:
    elegido_p2 = criterio_3.loc[criterio_3['compresion_%'].idxmin()]
    print(f"\nEscogemos el de MAYOR compresion con perdida < 3pp:")
else:
    # Si ninguno cumple 3pp, probamos con 5pp
    criterio_5 = df_p2[(df_p2['diff_cv'] >= -0.05) & (df_p2['diff_test'] >= -0.05)]
    elegido_p2 = criterio_5.loc[criterio_5['compresion_%'].idxmin()]
    print(f"\nNinguno cumple 3pp, escogemos con perdida < 5pp:")

n_elegido_p2 = int(elegido_p2['n_componentes'])
print(elegido_p2.to_string())
print(f"\n==> n_componentes elegido parte 2: {n_elegido_p2}")

Con los resultados de la tabla se ve bastante claro que a más componentes mejor funciona el modelo (como era de esperar), pero la gracia de la PCA es encontrar el punto donde podemos comprimir lo máximo posible sin perder demasiado rendimiento.

El código de arriba selecciona automáticamente el número de componentes que cumple el criterio de pérdida inferior a 3 puntos porcentuales tanto en CV como en test, priorizando la mayor compresión posible. Si ninguno llega al criterio de 3pp se relaja a 5pp.

Eso significa que ese número de componentes sería el que instalaríamos en las microcámaras para comprimir las imágenes antes de enviarlas al servidor central.

### #3 COMPRESION PARA CLASIFICACION POSTERIOR

**Objetivo**: Obtener el número de componentes que permita una compresión menor y al tiempo que el modelo en el servidor central no baje su rendimiento respecto a no usar imágenes comprimidas.

Para esta parte la idea que se os ha ocurrido es emplear también la PCA como compresor ya que así siempre podrían pasar a la opción anterior si eso fuese suficiente. Pero en este caso no vamos a utilizar el dataset comprimido con las PCAs para detectar las caras, sino el dataset una vez descomprimido (recuerda que puede emplear `inverse_transform` para "descomprimir"). Los pasos a seguir son:

1. Escoge un rango de valores que  permitan una compresión aún mayor (recuerda que el ancho de banda es mínimo) entre el 1 por mil y el 1 por ciento. Escoge 5 valores de número de PCAs que permitan movernos en ese rango.
2. Para cada uno de esos valores: aplica la PCA al X_train, obten un X_train_unzipped aplicando la inversa de la PCA y entrena un modelo de clasificación y pruébalo contra test, apunta el balanced accuracy.
3. Crea un dataframe o haz un visualización comparando como es la medidad de balance accuracy para cada valor de número de pcas escogido y cuál su factor de compresión. 
4. Sabiendo que no podemos perder más de 3 puntos porcentuales respecto al baseline, ¿qué numero de PCA escogerías?

In [ ]:
# Rango pedido: entre 0.1% (1 por mil) y 1% del tamanho original
# 0.1% de 4096 = 4.096 -> ~4 componentes
# 1%   de 4096 = 40.96 -> ~41 componentes

print(f"0.1% (1 por mil) de {n_features} = {n_features * 0.001:.1f} -> ~4 componentes")
print(f"1%               de {n_features} = {n_features * 0.010:.1f} -> ~41 componentes")

n_components_p3 = [4, 8, 15, 25, 40]
print(f"\nValores a probar: {n_components_p3}")
print(f"% del original:   {[round(n/n_features*100, 3) for n in n_components_p3]}")

In [ ]:
# Loop con inverse_transform: comprimimos -> enviamos -> descomprimimos -> clasificamos
# La idea es que la microcamara envia los datos comprimidos (X_comp)
# y el servidor los descomprime (X_rec) antes de clasificar

resultados_p3 = []

for n in n_components_p3:
    pca_n = PCA(n_components=n)

    # Comprimir (ajustamos sobre train)
    X_tr_comp = pca_n.fit_transform(X_train_scaled)
    X_te_comp = pca_n.transform(X_test_scaled)

    # Descomprimir con inverse_transform
    X_tr_rec = pca_n.inverse_transform(X_tr_comp)  # X_train_unzipped
    X_te_rec = pca_n.inverse_transform(X_te_comp)  # X_test_unzipped

    # Entrenamos y evaluamos sobre los datos DESCOMPRIMIDOS
    rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf.fit(X_tr_rec, y_train)
    test_score = balanced_accuracy_score(y_test, rf.predict(X_te_rec))

    compresion = (n / n_features) * 100

    resultados_p3.append({
        'n_componentes': n,
        'test_bal_acc':  round(test_score, 4),
        'compresion_%':  round(compresion, 4),
        'diff_baseline': round(test_score - test_baseline, 4)
    })
    print(f"  n={n:2d} | Test={test_score:.4f} | Compr={compresion:.3f}% | Diff={test_score - test_baseline:+.4f}")

In [ ]:
# Resultados en DataFrame y visualizacion
df_p3 = pd.DataFrame(resultados_p3)
print("Tabla de resultados - Parte 3:")
display(df_p3)

# Grafico comparativo con doble eje
fig, ax1 = plt.subplots(figsize=(10, 5))

ax1.plot(df_p3['n_componentes'], df_p3['test_bal_acc'],
         'o-', color='steelblue', linewidth=2, markersize=8, label='balanced_acc test')
ax1.axhline(test_baseline, color='red', linestyle='--',
            label=f'Baseline ({test_baseline:.4f})')
ax1.axhline(test_baseline - 0.03, color='orange', linestyle=':',
            label='Baseline - 3 pp (limite)')
ax1.set_xlabel('Numero de componentes PCA')
ax1.set_ylabel('Balanced Accuracy', color='steelblue')
ax1.tick_params(axis='y', labelcolor='steelblue')
ax1.set_title('Parte 3: balanced_accuracy con inverse_transform por numero de PCs')
ax1.legend(loc='lower right')

ax2 = ax1.twinx()
ax2.bar(df_p3['n_componentes'], df_p3['compresion_%'],
        alpha=0.25, color='green', label='% tamanho original')
ax2.set_ylabel('% del tamanho original', color='green')
ax2.tick_params(axis='y', labelcolor='green')

plt.tight_layout()
plt.show()

En esta parte la clave es el `inverse_transform`: comprimimos con la PCA, enviamos los datos comprimidos al servidor y allí los descomprimimos antes de clasificar. Así reducimos el ancho de banda pero la clasificación se hace sobre imágenes reconstruidas (no sobre las componentes directamente como en la parte 2).

El inconveniente es que al descomprimir perdemos algo de información (la reconstrucción no es perfecta con pocos componentes), y eso puede afectar al modelo.

In [ ]:
# Seleccion del n_components para parte 3
# Buscamos mayor compresion con perdida < 3pp respecto al baseline en test
criterio_3_p3 = df_p3[df_p3['diff_baseline'] >= -0.03]

if len(criterio_3_p3) > 0:
    elegido_p3 = criterio_3_p3.loc[criterio_3_p3['compresion_%'].idxmin()]
    n_elegido_p3 = int(elegido_p3['n_componentes'])
    print("Escogemos el de MAYOR compresion con perdida < 3pp:")
    print(elegido_p3.to_string())
    print(f"\n==> n_componentes elegido parte 3: {n_elegido_p3}")
else:
    print("Ninguno de los valores probados cumple el criterio de 3pp.")
    print("Habria que probar con mas componentes (por encima de 40).")
    mejor_p3 = df_p3.loc[df_p3['diff_baseline'].idxmax()]
    print(f"El de menor perdida es con {mejor_p3['n_componentes']} componentes: diff={mejor_p3['diff_baseline']:.4f}")

### #EXTRA

1. Para la segunda parte, visualiza en cuatro gráficos un scatter plot de las dos primeras componentes principales de la PCA escogida y colorea cada punto con las clases correspondientes a cada cara (como hay 40 clases, usa 10 por gráfico, 1-10 en el primero, 11-20 en el segundo, etc)
2. Para la tercer parte crea una función (modifica la de la práctica de la unidad de KMeans, por ejemplo) que permita ver la cara sin comprimir y la cara después de haberla descomprimido y haz una comprobación de cómo quedan (visualiza 5 caras por ejemplo) para cada uno de los valores de números de PCAs probados. Añade el caso para 150 y 320 PCs para que se vea que son las mismas claras con claridad.

In [ ]:
# EXTRA 1: Scatter de las 2 primeras PCs de la PCA de la parte 2
# Usamos X_train_pca que ya tiene todas las componentes calculadas
# Pintamos las dos primeras columnas (PC1 y PC2)

grupos = [(0, 10), (10, 20), (20, 30), (30, 40)]
titulos = ['Clases 1-10', 'Clases 11-20', 'Clases 21-30', 'Clases 31-40']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, (inicio, fin), titulo in zip(axes.flat, grupos, titulos):
    mask = (y_train >= inicio) & (y_train < fin)
    sc = ax.scatter(
        X_train_pca[mask, 0],
        X_train_pca[mask, 1],
        c=y_train[mask] % 10,  # % 10 para que el colormap use 10 colores distintos
        cmap='tab10',
        alpha=0.8,
        s=70
    )
    ax.set_title(titulo, fontsize=12)
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    plt.colorbar(sc, ax=ax, label='Clase (mod 10)')

plt.suptitle('Extra 1: Scatter de PC1 vs PC2 por grupos de clases', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# Se puede ver como algunas personas se separan bien (clusters claros)
# y otras estan mas mezcladas entre si

In [ ]:
# EXTRA 2: Funcion para comparar cara original vs reconstruida

def mostrar_caras_reconstruidas(X_scaled, n_components, n_caras=5):
    """
    Muestra n_caras comparando la imagen escalada original
    con la imagen reconstruida despues de comprimir/descomprimir con PCA.
    """
    pca = PCA(n_components=n_components)
    pca.fit(X_scaled)
    X_comp = pca.transform(X_scaled)
    X_rec  = pca.inverse_transform(X_comp)

    fig, axes = plt.subplots(2, n_caras, figsize=(n_caras * 2.5, 5))
    for i in range(n_caras):
        # Fila superior: cara original
        axes[0, i].imshow(X_scaled[i].reshape(64, 64), cmap='gray')
        axes[0, i].set_title('Original', fontsize=8)
        axes[0, i].axis('off')
        # Fila inferior: cara reconstruida
        axes[1, i].imshow(X_rec[i].reshape(64, 64), cmap='gray')
        axes[1, i].set_title(f'Recons.\n({n_components} PCs)', fontsize=8)
        axes[1, i].axis('off')

    varianza_acum = pca.explained_variance_ratio_.sum()
    plt.suptitle(
        f'n_components={n_components} | Varianza explicada: {varianza_acum:.3f}',
        fontsize=11
    )
    plt.tight_layout()
    plt.show()


# Probamos para los valores de la parte 3 y ademas 150 y 320
valores_extra = n_components_p3 + [150, 320]
print(f"Probando con: {valores_extra} componentes\n")

for n in valores_extra:
    mostrar_caras_reconstruidas(X_train_scaled, n_components=n, n_caras=5)